In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas as pd

df = pd.read_csv("/kaggle/input/titanic/train.csv")

print(df.head())

In [ ]:
df.shape


In [ ]:
sample = pd.read_csv("/kaggle/input/titanic/gender_submission.csv")

sample.head()

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
import xgboost as xgb
import optuna
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Load the data
train_df = pd.read_csv('/kaggle/input/titanic/train.csv')
test_df = pd.read_csv('/kaggle/input/titanic/test.csv')
sample_submission = pd.read_csv('/kaggle/input/titanic/gender_submission.csv')

# Data preprocessing function
def preprocess_data(df):
    df = df.copy()
    
    # Extract title from name
    df['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)
    df['Title'] = df['Title'].replace(['Lady', 'Countess','Capt', 'Col', 'Don', 'Dr', 
                                      'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')
    df['Title'] = df['Title'].replace('Mlle', 'Miss')
    df['Title'] = df['Title'].replace('Ms', 'Miss')
    df['Title'] = df['Title'].replace('Mme', 'Mrs')
    
    # Family size
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['IsAlone'] = (df['FamilySize'] == 1).astype(int)
    
    # Age filling
    df['Age'] = df.groupby(['Pclass', 'Sex', 'Title'])['Age'].transform(
        lambda x: x.fillna(x.median())
    )
    df['Age'].fillna(df['Age'].median(), inplace=True)
    
    # Fare filling
    df['Fare'].fillna(df['Fare'].median(), inplace=True)
    
    # Cabin features
    df['HasCabin'] = (~df['Cabin'].isna()).astype(int)
    df['Deck'] = df['Cabin'].str[0].fillna('Unknown')
    
    # Embarked filling
    df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)
    
    # Drop unnecessary columns
    df.drop(['PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1, inplace=True, errors='ignore')
    
    return df

# Preprocess data
print("Preprocessing data...")
train_processed = preprocess_data(train_df)
test_processed = preprocess_data(test_df)

# Encode categorical variables
label_encoders = {}
categorical_columns = ['Sex', 'Embarked', 'Title', 'Deck']

for col in categorical_columns:
    le = LabelEncoder()
    # Fit on train + test to handle all categories
    combined = pd.concat([train_processed[col], test_processed[col]], axis=0)
    le.fit(combined)
    train_processed[col] = le.transform(train_processed[col])
    test_processed[col] = le.transform(test_processed[col])
    label_encoders[col] = le

# Prepare features and target
X = train_processed.drop('Survived', axis=1)
y = train_processed['Survived']
X_test = test_processed

print(f"Training data shape: {X.shape}")
print(f"Test data shape: {X_test.shape}")

# Optuna objective function
def objective(trial):
    params = {
        'objective': 'binary:logistic',
        'eval_metric': 'logloss',
        'tree_method': 'hist',
        'random_state': 42,
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 1),
        'reg_lambda': trial.suggest_float('reg_lambda', 0, 1),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
    }
    
    # K-Fold Cross Validation
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    scores = []
    
    for train_idx, val_idx in kf.split(X, y):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        model = xgb.XGBClassifier(**params)
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            verbose=False,
            early_stopping_rounds=50
        )
        
        y_pred = model.predict(X_val)
        score = accuracy_score(y_val, y_pred)
        scores.append(score)
    
    return np.mean(scores)

# Optimize with Optuna
print("Starting Optuna optimization with 20 trials...")
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20, show_progress_bar=True)

print(f"Best trial: {study.best_trial.value}")
print("Best parameters:")
for key, value in study.best_trial.params.items():
    print(f"  {key}: {value}")

# Train final model with best parameters
best_params = study.best_trial.params
best_params.update({
    'objective': 'binary:logistic',
    'eval_metric': 'logloss',
    'tree_method': 'hist',
    'random_state': 42,
})

print("Training final model with best parameters...")
final_model = xgb.XGBClassifier(**best_params)
final_model.fit(X, y)

# Make predictions on test set
print("Making predictions on test set...")
test_predictions = final_model.predict(X_test)

# Create submission file
submission = pd.DataFrame({
    'PassengerId': test_df['PassengerId'],
    'Survived': test_predictions
})

# Save submission
submission.to_csv('submission.csv', index=False)
print("Submission file saved as 'submission.csv'")

# Display first few rows of submission
print("\nFirst 5 rows of submission:")
print(submission.head())

# Feature importance
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': final_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 10 most important features:")
print(feature_importance.head(10))

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import VotingClassifier, StackingClassifier
import xgboost as xgb
import lightgbm as lgb
import catboost as cb
import optuna
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# =====================
# Load Data
# =====================
print("Loading data...")
train_df = pd.read_csv('/kaggle/input/titanic/train.csv')
test_df = pd.read_csv('/kaggle/input/titanic/test.csv')
sample_submission = pd.read_csv('/kaggle/input/titanic/gender_submission.csv')

# =====================
# Preprocessing Function
# =====================
def preprocess_data(df):
    df = df.copy()
    
    # Extract title
    df['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)
    df['Title'] = df['Title'].replace(['Lady','Countess','Capt','Col','Don','Dr',
                                       'Major','Rev','Sir','Jonkheer','Dona'], 'Rare')
    df['Title'] = df['Title'].replace({'Mlle':'Miss', 'Ms':'Miss', 'Mme':'Mrs'})
    
    # Family and Alone
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['IsAlone'] = (df['FamilySize'] == 1).astype(int)
    
    # Age
    df['Age'] = df.groupby(['Pclass','Sex','Title'])['Age'].transform(lambda x: x.fillna(x.median()))
    df['Age'].fillna(df['Age'].median(), inplace=True)
    df['AgeGroup'] = pd.cut(df['Age'], bins=[0,12,18,35,60,100], labels=['Child','Teen','Adult','Middle','Senior'])
    
    # Fare
    df['Fare'].fillna(df['Fare'].median(), inplace=True)
    
    # Cabin
    df['HasCabin'] = (~df['Cabin'].isna()).astype(int)
    df['Deck'] = df['Cabin'].str[0].fillna('Unknown')
    
    # Embarked
    df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)
    
    # Ticket
    df['TicketPrefix'] = df['Ticket'].apply(lambda x: str(x).split()[0] if len(str(x).split()) > 1 else 'NUM')
    df['TicketPrefix'] = df['TicketPrefix'].apply(lambda x: x if x.isalpha() else 'NUM')
    
    # Drop unused
    df.drop(['PassengerId','Name','Ticket','Cabin'], axis=1, inplace=True, errors='ignore')
    return df

# =====================
# Process Data
# =====================
print("Preprocessing data...")
train_processed = preprocess_data(train_df)
test_processed = preprocess_data(test_df)
test_passenger_ids = test_df['PassengerId'].copy()

# Label encoding
categorical_cols = ['Sex','Embarked','Title','Deck','AgeGroup','TicketPrefix']
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    combined = pd.concat([train_processed[col], test_processed[col]], axis=0)
    le.fit(combined)
    train_processed[col] = le.transform(train_processed[col])
    test_processed[col] = le.transform(test_processed[col])
    label_encoders[col] = le

# Split X, y
X = train_processed.drop('Survived', axis=1)
y = train_processed['Survived']
X_test = test_processed

# Ensure same cols
missing_cols = set(X.columns) - set(X_test.columns)
for col in missing_cols:
    X_test[col] = 0
X_test = X_test[X.columns]

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_test_scaled = scaler.transform(X_test)

X_scaled = pd.DataFrame(X_scaled, columns=X.columns, index=X.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

# =====================
# Optuna Optimization (no early stopping)
# =====================
def optimize_xgboost(trial):
    params = {
        'n_estimators': 500,
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2),
        'subsample': trial.suggest_float('subsample', 0.7, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 1),
        'reg_lambda': trial.suggest_float('reg_lambda', 0, 1),
        'random_state': 42,
        'use_label_encoder': False
    }
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    scores = []
    for tr, val in kf.split(X_scaled, y):
        model = xgb.XGBClassifier(**params)
        model.fit(X_scaled.iloc[tr], y.iloc[tr])
        preds = model.predict(X_scaled.iloc[val])
        scores.append(accuracy_score(y.iloc[val], preds))
    return np.mean(scores)

def optimize_lightgbm(trial):
    params = {
        'n_estimators': 500,
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2),
        'subsample': trial.suggest_float('subsample', 0.7, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 1),
        'reg_lambda': trial.suggest_float('reg_lambda', 0, 1),
        'num_leaves': trial.suggest_int('num_leaves', 20, 50),
        'random_state': 42
    }
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    scores = []
    for tr, val in kf.split(X_scaled, y):
        model = lgb.LGBMClassifier(**params)
        model.fit(X_scaled.iloc[tr], y.iloc[tr])
        preds = model.predict(X_scaled.iloc[val])
        scores.append(accuracy_score(y.iloc[val], preds))
    return np.mean(scores)

def optimize_catboost(trial):
    params = {
        'iterations': 500,
        'depth': trial.suggest_int('depth', 4, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 5),
        'random_strength': trial.suggest_float('random_strength', 0, 1),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0, 1),
        'random_state': 42,
        'verbose': False
    }
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    scores = []
    for tr, val in kf.split(X, y):
        model = cb.CatBoostClassifier(**params)
        model.fit(X.iloc[tr], y.iloc[tr])
        preds = model.predict(X.iloc[val])
        scores.append(accuracy_score(y.iloc[val], preds))
    return np.mean(scores)

# Run Optuna
print("\nOptimizing models with Optuna (no early stopping)...")
study_xgb = optuna.create_study(direction='maximize')
study_xgb.optimize(optimize_xgboost, n_trials=10, show_progress_bar=True)

study_lgb = optuna.create_study(direction='maximize')
study_lgb.optimize(optimize_lightgbm, n_trials=10, show_progress_bar=True)

study_cb = optuna.create_study(direction='maximize')
study_cb.optimize(optimize_catboost, n_trials=10, show_progress_bar=True)

best_xgb_params = study_xgb.best_params
best_lgb_params = study_lgb.best_params
best_cb_params = study_cb.best_params

print("\nBest XGBoost:", best_xgb_params)
print("Best LightGBM:", best_lgb_params)
print("Best CatBoost:", best_cb_params)

# =====================
# Train Final Models
# =====================
xgb_model = xgb.XGBClassifier(**best_xgb_params, n_estimators=500, use_label_encoder=False, random_state=42)
lgb_model = lgb.LGBMClassifier(**best_lgb_params, n_estimators=500, random_state=42)
cb_model = cb.CatBoostClassifier(**best_cb_params, iterations=500, random_state=42, verbose=False)

rf_model = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42)
gb_model = GradientBoostingClassifier(n_estimators=200, random_state=42)
logreg_model = LogisticRegression(max_iter=1000, random_state=42)

voting_clf = VotingClassifier(
    estimators=[('xgb', xgb_model), ('lgb', lgb_model), ('cb', cb_model), ('rf', rf_model)],
    voting='soft'
)

stacking_clf = StackingClassifier(
    estimators=[('xgb', xgb_model), ('lgb', lgb_model), ('cb', cb_model)],
    final_estimator=LogisticRegression(),
    cv=3
)

models = {
    'XGBoost': xgb_model,
    'LightGBM': lgb_model,
    'CatBoost': cb_model,
    'RandomForest': rf_model,
    'GradientBoosting': gb_model,
    'LogisticRegression': logreg_model,
    'Voting': voting_clf,
    'Stacking': stacking_clf
}

# =====================
# Cross-validation
# =====================
print("\nEvaluating models...")
cv_results = {}
kf = KFold(n_splits=5, shuffle=True, random_state=42)
for name, model in models.items():
    try:
        scores = cross_val_score(model, X_scaled, y, cv=kf, scoring='accuracy')
        cv_results[name] = scores.mean()
        print(f"{name}: {scores.mean():.4f}")
    except Exception as e:
        print(f"{name} failed: {e}")
        cv_results[name] = 0

best_model_name = max(cv_results, key=cv_results.get)
print(f"\nBest model: {best_model_name}")

best_model = models[best_model_name]
best_model.fit(X_scaled, y)
test_preds = best_model.predict(X_test_scaled)

# =====================
# Weighted Ensemble
# =====================
print("\nCreating weighted ensemble...")
top_models = sorted(cv_results.items(), key=lambda x: x[1], reverse=True)[:3]
weights = [s for _, s in top_models]
weights = [w/sum(weights) for w in weights]

ensemble_pred = np.zeros(len(X_test))
for (name, score), w in zip(top_models, weights):
    model = models[name]
    model.fit(X_scaled, y)
    preds = model.predict_proba(X_test_scaled)[:, 1]
    ensemble_pred += preds * w

final_preds = (ensemble_pred > 0.5).astype(int)

# =====================
# Submission
# =====================
submission = pd.DataFrame({'PassengerId': test_passenger_ids, 'Survived': final_preds})
submission.to_csv('submission.csv', index=False)
print("\nSubmission file 'submission.csv' saved successfully!")
print(submission.head(10))
